In [93]:
# Initialize Otter
import otter
grader = otter.Notebook("hw8.ipynb")

# CPSC 330 - Applied Machine Learning

## Homework 8: Introduction to Computer vision and Time Series

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [94]:
from hashlib import sha1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>

_Points:_ 2

<!-- END QUESTION -->

<br><br>

## Exercise 1: time series prediction

In this exercise we'll be looking at a [dataset of avocado prices](https://www.kaggle.com/neuromusic/avocado-prices). You should start by downloading the dataset and storing it under the `data` folder. We will be forcasting average avocado price for the next week. 

In [95]:
df = pd.read_csv("data/avocado.csv", parse_dates=["Date"], index_col=0)
df.head()

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany


In [96]:
df.shape

(18249, 13)

In [97]:
df["Date"].min()

Timestamp('2015-01-04 00:00:00')

In [98]:
df["Date"].max()

Timestamp('2018-03-25 00:00:00')

It looks like the data ranges from the start of 2015 to March 2018 (~2 years ago), for a total of 3.25 years or so. Let's split the data so that we have a 6 months of test data.

In [99]:
split_date = '20170925'
df_train = df[df["Date"] <= split_date]
df_test  = df[df["Date"] >  split_date]

In [100]:
assert len(df_train) + len(df_test) == len(df)

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 How many time series? 
rubric={points:4}

In the [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) dataset from lecture demo, we had different measurements for each Location. 

We want you to consider this for the avocado prices dataset. For which categorical feature(s), if any, do we have separate measurements? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 4

Upon inspecting the dataframe, we can see that there are 54 unique region values and 2 type values. The PLU numbers only correspond to a specific size category of avocadoes and thus are co-variates with our target variable of AveragePrice. Similarly, bag sizes refer to how the avocadoes were sold describing the composition of the sales figure, which may help predict the price but is also a co-variate, not its own category. Thus our final tally of number of time series data present in this dataframe is 108 given by each (region, type) pair, with `region` and `type` being the categorical features for which we have separate measurements.

In [101]:
df_train.describe(include='all')

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
count,15441,15441.000000,1.544100e+04,1.544100e+04,1.544100e+04,1.544100e+04,1.544100e+04,1.544100e+04,1.544100e+04,15441.000000,15441,15441.000000,15441
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,54
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,conventional,NaN,Albany
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7722,NaN,286
mean,2016-05-14 22:59:56.502817,1.397126,8.415280e+05,2.915050e+05,2.987060e+05,2.438154e+04,2.269345e+05,1.739725e+05,4.997246e+04,2989.486697,NaN,2015.909008,NaN
min,2015-01-04 00:00:00,0.440000,8.456000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,NaN,2015.000000,NaN
25%,2015-09-06 00:00:00,1.080000,1.009291e+04,8.790700e+02,3.114340e+03,0.000000e+00,4.072320e+03,2.087380e+03,1.066700e+02,0.000000,NaN,2015.000000,NaN
50%,2016-05-15 00:00:00,1.360000,1.044206e+05,8.217170e+03,2.983921e+04,1.987000e+02,3.732293e+04,2.419380e+04,2.348730e+03,0.000000,NaN,2016.000000,NaN
75%,2017-01-22 00:00:00,1.660000,4.273913e+05,1.113626e+05,1.518530e+05,7.051080e+03,1.035380e+05,7.913568e+04,1.952399e+04,106.940000,NaN,2017.000000,NaN
max,2017-09-24 00:00:00,3.250000,6.103446e+07,2.274362e+07,2.047057e+07,2.546439e+06,1.629830e+07,1.256716e+07,4.324231e+06,551693.650000,NaN,2017.000000,NaN


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Equally spaced measurements? 
rubric={points:4}

In the Rain in Australia dataset, the measurements were generally equally spaced but with some exceptions. How about with this dataset? Justify your answer by referencing the dataset.

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 4

Conducting an initial analysis of the gaps normally between dates, we see that there are a lot of rows that share the same date, with a few having a 7-day gap which constitutes our actual time series data. However, upon noting that we have 108 different time series present in our dataframe, we can perform a more granular analysis by grouping by our selected categorical variables of `region` and `type` and checking for differences within it. The result of this aligns with our expectations as there is no longer a '0 days' count row entry and we see that most of the data has even 7 day differences, with exactly 3 missing data entries somewhere in some time series, where in one we miss a single week (to get a 14 day difference) and in another (or potentially same time series) we get a 2 week gap with no data. It is still overall safe to say that we have mostly equally spaced data with very few gaps.

In [102]:
df_train["Date"].sort_values().diff().value_counts()

Date
0 days    15298
7 days      142
Name: count, dtype: int64

In [103]:
df_train_sorted = df_train.sort_values("Date")

date_diffs = (
    df_train_sorted
    .groupby(["region", "type"])["Date"]
    .diff()
    .value_counts()
)

print(date_diffs)

Date
7 days     15331
14 days        1
21 days        1
Name: count, dtype: int64


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Interpreting regions 
rubric={points:4}

In the Rain in Australia dataset, each location was a different place in Australia. For this dataset, look at the names of the regions. Do you think the regions are also all distinct, or are there overlapping regions? Justify your answer by referencing the data.

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

When we inspect the region names, we notice entries like (1) Large regions like `California`, `West`, `Northeast`, `Plains`, etc. which clearly encompass other regions like granular city names, (2) Aggregated multi-city regions like `MiamiFtLauderdale` and `DallasFtWorth`, (3) National aggregate `TotalUS` which obviously includes all other regions for this dataset. Thus, we can conclude that the regions are not distinct and very much do have overlapping geographical groupings.

In [104]:
df_train["region"].unique()

<StringArray>
[             'Albany',             'Atlanta', 'BaltimoreWashington',
               'Boise',              'Boston',    'BuffaloRochester',
          'California',           'Charlotte',             'Chicago',
    'CincinnatiDayton',            'Columbus',       'DallasFtWorth',
              'Denver',             'Detroit',         'GrandRapids',
          'GreatLakes',  'HarrisburgScranton', 'HartfordSpringfield',
             'Houston',        'Indianapolis',        'Jacksonville',
            'LasVegas',          'LosAngeles',          'Louisville',
   'MiamiFtLauderdale',            'Midsouth',           'Nashville',
    'NewOrleansMobile',             'NewYork',           'Northeast',
  'NorthernNewEngland',             'Orlando',        'Philadelphia',
       'PhoenixTucson',          'Pittsburgh',              'Plains',
            'Portland',   'RaleighGreensboro',     'RichmondNorfolk',
             'Roanoke',          'Sacramento',            'SanDiego',
      

<!-- END QUESTION -->

<br><br>

We will use the entire dataset despite any location-based weirdness uncovered in the previous part.

We will be trying to forecast the avocado price. The function below is adapted from the lecture.

In [105]:
import pandas as pd


def create_lag_feature(
    df: pd.DataFrame,
    orig_feature: str,
    lag: int,
    groupby: list[str],
    new_feature_name: str | None = None,
    clip: bool = False,
) -> pd.DataFrame:
    """
    Create a lagged (or ahead) version of a feature, optionally per group.

    Assumes df is already sorted by time within each group and has unique indices.

    Parameters
    ----------
    df : pd.DataFrame
        The dataset.
    orig_feature : str
        Name of the column to lag.
    lag : int
        The lag:
          - negative → values from the past (t-1, t-2, ...)
          - positive → values from the future (t+1, t+2, ...)
    groupby : list of str
        Column(s) to group by if df contains multiple time series.
    new_feature_name : str, optional
        Name of the new column. If None, a name is generated automatically.
    clip : bool, default False
        If True, drop rows where the new feature is NaN.

    Returns
    -------
    pd.DataFrame
        A new dataframe with the additional column added.
    """
    if lag == 0:
        raise ValueError("lag cannot be 0 (no shift). Use the original feature instead.")

    # Default name if not provided
    if new_feature_name is None:
        if lag < 0:
            new_feature_name = f"{orig_feature}_lag{abs(lag)}"
        else:
            new_feature_name = f"{orig_feature}_ahead{lag}"

    df = df.copy()

    # Map your convention (negative=past, positive=future) to pandas shift
    # pandas: shift(+k) → past, shift(-k) → future
    periods = abs(lag) if lag < 0 else -lag

    df[new_feature_name] = (
        df.groupby(groupby, sort=False)[orig_feature]
          .shift(periods)
    )

    if clip:
        df = df.dropna(subset=[new_feature_name])

    return df


We first sort our dataframe properly:

In [106]:
df_sort = df.sort_values(by=["region", "type", "Date"]).reset_index(drop=True)
df_sort

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico
18247,2018-03-18,1.56,15896.38,2055.35,1499.55,0.00,12341.48,12114.81,226.67,0.0,organic,2018,WestTexNewMexico


We then call `create_lag_feature`. This creates a new column in the dataset `AveragePriceNextWeek`, which is the following week's `AveragePrice`. We have set `clip=True` which means it will remove rows where the target would be missing.

In [107]:
df_lagged = create_lag_feature(df_sort, "AveragePrice", -1, ["region", "type"], "lag1")
df_lagged = create_lag_feature(df_lagged, "AveragePrice", -2, ["region", "type"], "lag2")
df_lagged = create_lag_feature(df_lagged, "AveragePrice", -3, ["region", "type"], "lag3")

df_hastarget = create_lag_feature(df_lagged, "AveragePrice", +1, ["region", "type"], "AveragePriceNextWeek", clip=True)
df_hastarget

,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region,lag1,lag2,lag3,AveragePriceNextWeek
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany,NaN,NaN,NaN,1.24
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany,1.22,NaN,NaN,1.17
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany,1.24,1.22,NaN,1.06
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany,1.17,1.24,1.22,0.99
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany,1.06,1.17,1.24,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18243,2018-02-18,1.56,17597.12,1892.05,1928.36,0.00,13776.71,13553.53,223.18,0.0,organic,2018,WestTexNewMexico,1.57,1.63,1.71,1.57
18244,2018-02-25,1.57,18421.24,1974.26,2482.65,0.00,13964.33,13698.27,266.06,0.0,organic,2018,WestTexNewMexico,1.56,1.57,1.63,1.54
18245,2018-03-04,1.54,17393.30,1832.24,1905.57,0.00,13655.49,13401.93,253.56,0.0,organic,2018,WestTexNewMexico,1.57,1.56,1.57,1.56
18246,2018-03-11,1.56,22128.42,2162.67,3194.25,8.93,16762.57,16510.32,252.25,0.0,organic,2018,WestTexNewMexico,1.54,1.57,1.56,1.56


Our goal is to predict `AveragePriceNextWeek`. 

Let's split the data:

In [108]:
df_train = df_hastarget[df_hastarget["Date"] <= split_date]
df_test  = df_hastarget[df_hastarget["Date"] >  split_date]

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 `AveragePrice` baseline 
rubric={points}

Soon we will want to build some models to forecast the average avocado price a week in advance. Before we start with any ML though, let's try a baseline. Previously we used `DummyClassifier` or `DummyRegressor` as a baseline. This time, we'll do something else as a baseline: we'll assume the price stays the same from this week to next week. So, we'll set our prediction of "AveragePriceNextWeek" exactly equal to "AveragePrice", assuming no change. That is kind of like saying, "If it's raining today then I'm guessing it will be raining tomorrow". This simplistic approach will not get a great score but it's a good starting point for reference. If our model does worse that this, it must not be very good. 

Using this baseline approach, what $R^2$ do you get on the train and test data?

<div class="alert alert-warning">

Solution_1.4
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [109]:
y_train = df_train["AveragePriceNextWeek"]
y_test  = df_test["AveragePriceNextWeek"]

y_train_pred = df_train["AveragePrice"]
y_test_pred  = df_test["AveragePrice"]

In [110]:
train_r2 = r2_score(y_train, y_train_pred)
train_r2

0.8285800937261841

In [111]:
test_r2  = r2_score(y_test, y_test_pred)
test_r2

0.7631780188583048

In [112]:
assert not train_r2 is None, "Are you using the correct variable name?"
assert not test_r2 is None, "Are you using the correct variable name?"
assert sha1(str(round(train_r2, 3)).encode('utf8')).hexdigest() == 'b1136fe2a8918904393ab6f40bfb3f38eac5fc39', "Your training score is not correct. Are you using the right features?"
assert sha1(str(round(test_r2, 3)).encode('utf8')).hexdigest() == 'cc24d9a9b567b491a56b42f7adc582f2eefa5907', "Your test score is not correct. Are you using the right features?"

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Forecasting average avocado price
rubric={points:10}

Now that the baseline is done, let's build some models to forecast the average avocado price a week later. Experiment with a few approachs for encoding the date. Justify the decisions you make. Which approach worked best? Report your test score and briefly discuss your results.

Benchmark: you should be able to achieve $R^2$ of at least 0.79 on the test set. I got to 0.80, but not beyond that. Let me know if you do better!

Note: because we only have 2 splits here, we need to be a bit wary of overfitting on the test set. Try not to test on it a ridiculous number of times. If you are interested in some proper ways of dealing with this, see for example sklearn's [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), which is like cross-validation for time series data.

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Points:_ 10

I first chose to utilize a Ridge model on all the features which gave me a baseline of R^2=0.7732. Then interestingly, removing features by trial and error, I found that only AveragePrice and PLU columns were generating useful signal as removing every other feature one at a time gave me a higher score. To try to push beyond this, I tried including week and month as features too, where week data was encoded in sin and cos versions so the model would understand that Mondays are close to Sundays, etc. These efforts only brought me up to a R^2 0.7827 so I switched to using a RandomForestRegressor and did similar feature selection by trial and error across the features developed so far, which this time brought me up to 0.7774 - surprisingly worse than the Ridge model. I then retroactively wrote in lag features so that more of the previous average prices could be used by the model as Average Price was generating the most signal, and this brought up our RandomForestRegressor to 0.8000 .

In [113]:
def trainRidgeOnAvocadoPrice(features):
    """
    Train Ridge model based on features 
    """
    X_train = df_train[features]
    X_test  = df_test[features]

    model = Ridge()
    model.fit(X_train, y_train)

    print("Test R2:", r2_score(y_test, model.predict(X_test)))

In [114]:
features = [
    "AveragePrice", "4046", "4225", "4770", "year",
    "Total Volume", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags"
]

trainRidgeOnAvocadoPrice(features)

Test R2: 0.7731920697216398


c:\Users\kreet\Desktop\Files\UBC\Year 4\Term 2\CPSC 330\HW\venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.269568990733989e-17.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [115]:
features = [
    "AveragePrice", "4046", "4225", "4770"
]

trainRidgeOnAvocadoPrice(features)

Test R2: 0.782538072289197


In [116]:
df_train["month"] = df_train["Date"].dt.month
df_test["month"] = df_test["Date"].dt.month

features_with_month = features + ["month"]

trainRidgeOnAvocadoPrice(features_with_month)

Test R2: 0.7824876546404392


In [117]:
for df_ in [df_train, df_test]:
    df_["weekofyear"] = df_["Date"].dt.isocalendar().week.astype(int)
    df_["week_sin"] = np.sin(2 * np.pi * df_["weekofyear"] / 52)
    df_["week_cos"] = np.cos(2 * np.pi * df_["weekofyear"] / 52)

features_cyclical = features + ["weekofyear"]

trainRidgeOnAvocadoPrice(features_cyclical)

Test R2: 0.7826913441482789


In [139]:
features = [
    "AveragePrice", "4046", "4225", "4770", 
    "Total Volume", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags", "week_sin", "week_cos"
]

X_train = df_train[features]
X_test  = df_test[features]

model = RandomForestRegressor(n_estimators=100, random_state=0)

model.fit(X_train, y_train)

print("Test R2:", r2_score(y_test, model.predict(X_test)))

Test R2: 0.7774606121888257


In [140]:
features = [
    "AveragePrice", "4046", "4225", "4770", 
    "Total Volume", "Total Bags", "Small Bags", "Large Bags", "XLarge Bags", "week_sin", "week_cos", "lag1", "lag2", "lag3"
]

X_train = df_train[features]
X_test  = df_test[features]

model = RandomForestRegressor(n_estimators=100, random_state=0)

model.fit(X_train, y_train)

print("Test R2:", r2_score(y_test, model.predict(X_test)))

Test R2: 0.8000679931938603


## Exercise 2: Short answer questions

<!-- BEGIN QUESTION -->

### 2.1 Time series

rubric={points:6}

The following questions pertain to Lecture 20 on time series data:

1. Sometimes a time series has missing time points or, worse, time points that are unequally spaced in general. Give an example of a real world situation where the time series data would have unequally spaced time points.
2. In class we discussed two approaches to using temporal information: encoding the date as one or more features, and creating lagged versions of features. Which of these (one/other/both/neither) two approaches would struggle with unequally spaced time points? Briefly justify your answer.
3. When studying time series modeling, we explored several ways to encode date information as a feature for the citibike dataset. When we used time of day as a numeric feature, the Ridge model was not able to capture the periodic pattern. Why? How did we tackle this problem? Briefly explain.

<div class="alert alert-warning">

Solution_2.1
    
</div>

_Points:_ 6

1. An example of time series data that may have unequally spaced or missing points could be any form of trading data that captures when transactions were made, as these transactions are free to occur at any uneven intervals.
2. Lagged features would struggle more, as k previous data points at uneven intervals could correspond to points that represent data captured very close to where we are predicted, or arbitrarily far. Thus, any potential pattern our model could hope to learn would have distortions due to abscissae being at different points. On the other hand, encoding date into more granular features would retain the exact time information which would avoid this distortion issue when the model tries to learn a pattern.
3. Because the Ridge model learns a linear mapping, there will be jumps from 23 hours to 0 hours as the model treats these as far apart. We can tackle this by using cyclical encoding that puts both the times at the edge closer in the feature space.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Computer vision 
rubric={points:6}

The following questions pertain to the lecture on multiclass classification and introduction to computer vision. 

1. How many parameters (coefficients and intercepts) will `sklearn`’s `LogisticRegression()` model learn for a four-class classification problem, assuming that you have 10 features? Briefly explain your answer.
2. In Lecture 19, we briefly discussed how neural networks are sort of like `sklearn`'s pipelines, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?
3. Imagine that you have a small dataset with ~1000 images containing pictures and names of 50 different Computer Science faculty members from UBC. Your goal is to develop a reasonably accurate multi-class classification model for this task. Describe which model/technique you would use and briefly justify your choice in one to three sentences.

<div class="alert alert-warning">

Solution_2.2
    
</div>

_Points:_ 6

1. A LogisticRegression model must learn 10 coefficients for each feature + 1 intercept, and it will do this for each class individually for multiclass default one-vs-rest classification, giving us 4 * (10+1) = 44 parameters.
2. Since multiple transformations are learned between layers, the initial and in-between layers effectively do feature transformation and extraction, and these early layers can then be reused in other similar downstream tasks that operate on similar data and thus need similar features extracted (e.g. edges and textures for images) with only fine-tuning for later layers then being required.
3. Our dataset is extremely small and thus training a model from scratch would likely lead to overfitting and not generalize well on new images. Thus, it is probably best to utilize a pre-trained CNN with transfer learning and only fine-tune the terminal few layers for our specific 50 classes.

<!-- END QUESTION -->

<br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)